In [1]:
!pip install elasticsearch googletrans==4.0.0-rc1 sentence-transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.2/571.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 2.3 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17395 sha256=9fdf0fa70598f50928e0254df9c3ba0889a0784bcb67c2619da0008bc790661b
  Stored 

In [2]:
!pip install -q elasticsearch

In [3]:
import json
import os
from elasticsearch import Elasticsearch
import numpy as np

In [4]:
import sys
sys.path.append('/kaggle/input/py-lib/AIC')

from elastic_search_processor import ElasticSearchProcessor

In [5]:
es_processor = ElasticSearchProcessor(
    "https://8641545ce73f41b3a05dbc80de48d72e.asia-northeast1.gcp.cloud.es.io:443", 
    "aHdlcWFaUUJnZjYwZ0FWNHN1Snk6THZFVl9XLS1Rb0tQcERyTkVCS08ydw==", 
    "huyrc")

success, message = es_processor.create_index_with_mapping()
print(message)

Index 'huyrc' đã tồn tại


In [6]:
es_processor.client.ping()

True

In [7]:
indexMapping = {
    "properties": {
        "video_id": {
            "type": "keyword"
        },
        "start_time": {
            "type": "float"
        },
        "end_time": {
            "type": "float"
        },
        "AudioTextVector": {
            "type": "dense_vector",
            "dims": 768,
            "index": True,
            "similarity": "l2_norm"
        }
    }
}
#response = es_processor.client.indices.delete(index='audio_features_1')
#response = es_processor.client.indices.delete(index='audio_features_2')
#response = es_processor.client.indices.delete(index='audio_features_3')
#response = es_processor.client.indices.delete(index='audio_features_4')

es_processor.client.indices.create(index="audio_features_1", mappings=indexMapping)
es_processor.client.indices.create(index="audio_features_2", mappings=indexMapping)
es_processor.client.indices.create(index="audio_features_3", mappings=indexMapping)
es_processor.client.indices.create(index="audio_features_4", mappings=indexMapping)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'audio_features_4'})

In [8]:
folder_path = '/kaggle/input/full-30l-audio-features/31 audio features'
i = 0
# Loop through the folder and load each .npy file
for file_name in os.listdir(folder_path):
    if file_name.endswith('.npy'):
        # Parse the file name to extract video ID, start time, and end time
        parts = file_name.replace('.npy', '').split('_')
        video_id = parts[0] + "_" + parts[1]  # e.g., L01_V001
        start_time = float(parts[2])  # Start time from filename
        end_time = float(parts[3])  # End time from filename

        # Load the corresponding feature vector from the .npy file
        file_path = os.path.join(folder_path, file_name)
        audio_text_vector = np.load(file_path).tolist()  # Convert numpy array to list

        # Create the document to be indexed
        doc = {
            "video_id": video_id,
            "start_time": start_time,
            "end_time": end_time,
            "AudioTextVector": audio_text_vector
        }
        if i < 10000:
            # Index the document in Elasticsearch
            try:
                es_processor.client.index(index="audio_features_1", document=doc)
                print(f"Indexed {file_name} successfully")
            except Exception as e:
                print(f"Error indexing {file_name}: {e}")
        if i >= 10000 and i < 20000:
            try:
                es_processor.client.index(index="audio_features_2", document=doc)
                print(f"Indexed {file_name} successfully")
            except Exception as e:
                print(f"Error indexing {file_name}: {e}")
        if i >= 20000 and i < 30000:
            try:
                es_processor.client.index(index="audio_features_3", document=doc)
                print(f"Indexed {file_name} successfully")
            except Exception as e:
                print(f"Error indexing {file_name}: {e}")
        if i >= 30000:
            try:
                es_processor.client.index(index="audio_features_4", document=doc)
                print(f"Indexed {file_name} successfully")
            except Exception as e:
                print(f"Error indexing {file_name}: {e}")
        i += 1

Indexed L26_V115_225_240.npy successfully
Indexed L25_V078_255_270.npy successfully
Indexed L25_V068_1170_1185.npy successfully
Indexed L29_V016_330_345.npy successfully
Indexed L26_V310_165_180.npy successfully
Indexed L30_V045_120_135.npy successfully
Indexed L11_V019_893.0_969.0.npy successfully
Indexed L29_V021_15_30.npy successfully
Indexed L26_V254_270_285.npy successfully
Indexed L26_V357_285_300.npy successfully
Indexed L20_V031_1107.0_1147.0.npy successfully
Indexed L25_V033_1770_1785.npy successfully
Indexed L01_V001_107.0_127.0.npy successfully
Indexed L19_V025_482.0_502.0.npy successfully
Indexed L26_V304_180_195.npy successfully
Indexed L26_V164_75_90.npy successfully
Indexed L25_V049_1515_1530.npy successfully
Indexed L07_V005_940.0_1010.0.npy successfully
Indexed L26_V234_75_90.npy successfully
Indexed L26_V430_255_270.npy successfully
Indexed L30_V092_15_30.npy successfully
Indexed L26_V008_105_120.npy successfully
Indexed L06_V010_778.0_798.0.npy successfully
Indexed L

In [9]:

from googletrans import Translator
from sentence_transformers import SentenceTransformer

# Initialize translator and sentence transformer model
translator = Translator()
model = SentenceTransformer("multi-qa-mpnet-base-cos-v1")

# Function to encode and normalize the text query
def encode_text_query(query):
    translated = translator.translate(query, dest='en')  # Translate to English if necessary
    query = translated.text
    feature_vector = np.array(model.encode(query))
    feature_vector = feature_vector[np.newaxis, :]
    feature_vector = feature_vector / np.linalg.norm(feature_vector)  # Normalize the vector
    return feature_vector

# Encode the input keyword
input_keyword = "tiệm sửa xe bốc cháy dữ dội"
vector = encode_text_query(input_keyword)
doc_count_1 = es_processor.client.count(index='audio_features_1')['count']
doc_count_2 = es_processor.client.count(index='audio_features_2')['count']
doc_count_3 = es_processor.client.count(index='audio_features_3')['count']
doc_count_4 = es_processor.client.count(index='audio_features_4')['count']

# KNN search query
query_1 = {
    "field": "AudioTextVector",
    "query_vector": vector.flatten().tolist(),  # Ensure it's a list for Elasticsearch
    "k": 10,  # Return top 10 results
    "num_candidates": doc_count_1
}
query_2 = {
    "field": "AudioTextVector",
    "query_vector": vector.flatten().tolist(),  # Ensure it's a list for Elasticsearch
    "k": 10,  # Return top 10 results
    "num_candidates": doc_count_2
}
query_3 = {
    "field": "AudioTextVector",
    "query_vector": vector.flatten().tolist(),  # Ensure it's a list for Elasticsearch
    "k": 10,  # Return top 10 results
    "num_candidates": doc_count_3
}
query_4 = {
    "field": "AudioTextVector",
    "query_vector": vector.flatten().tolist(),  # Ensure it's a list for Elasticsearch
    "k": 10,  # Return top 10 results
    "num_candidates": doc_count_4
}
# Perform the search
print("First part:")
res = es_processor.client.knn_search(index="audio_features_1", knn=query_1, _source=["video_id", "start_time", "end_time"])
# Output the search results
for hit in res["hits"]["hits"]:
    print(f"Video ID: {hit['_source']['video_id']}, Start: {hit['_source']['start_time']}, End: {hit['_source']['end_time']}")
print("Second part:")
res = es_processor.client.knn_search(index="audio_features_2", knn=query_2, _source=["video_id", "start_time", "end_time"])
# Output the search results
for hit in res["hits"]["hits"]:
    print(f"Video ID: {hit['_source']['video_id']}, Start: {hit['_source']['start_time']}, End: {hit['_source']['end_time']}")
print("Third_part:")
res = es_processor.client.knn_search(index="audio_features_3", knn=query_3, _source=["video_id", "start_time", "end_time"])
# Output the search results
for hit in res["hits"]["hits"]:
    print(f"Video ID: {hit['_source']['video_id']}, Start: {hit['_source']['start_time']}, End: {hit['_source']['end_time']}")
print("Fourth_part:")
res = es_processor.client.knn_search(index="audio_features_4", knn=query_4, _source=["video_id", "start_time", "end_time"])
# Output the search results
for hit in res["hits"]["hits"]:
    print(f"Video ID: {hit['_source']['video_id']}, Start: {hit['_source']['start_time']}, End: {hit['_source']['end_time']}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.25k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

/tmp/ipykernel_30/3013669256.py:57: ElasticsearchWarning: The kNN search API has been replaced by the `knn` option in the search API.
  res = es_processor.client.knn_search(index="audio_features_2", knn=query_2, _source=["video_id", "start_time", "end_time"])
/tmp/ipykernel_30/3013669256.py:62: GeneralAvailabilityWarning: This API is in technical preview and may be changed or removed in a future release. Elastic will work to fix any issues, but features in technical preview are not subject to the support SLA of official GA features.
  res = es_processor.client.knn_search(index="audio_features_3", knn=query_3, _source=["video_id", "start_time", "end_time"])


Video ID: L05_V018, Start: 523.0, End: 585.0
Video ID: L14_V025, Start: 1166.0, End: 1204.0
Video ID: L11_V014, Start: 359.0, End: 408.0
Video ID: L04_V026, Start: 1051.0, End: 1091.0
Video ID: L08_V011, Start: 979.0, End: 1029.0
Video ID: L04_V002, Start: 1131.0, End: 1166.0
Video ID: L13_V006, Start: 476.0, End: 521.0
Video ID: L14_V021, Start: 802.0, End: 838.0
Video ID: L14_V016, Start: 1232.0, End: 1281.0
Video ID: L22_V019, Start: 889.0, End: 937.0
Third_part:


/tmp/ipykernel_30/3013669256.py:62: ElasticsearchWarning: The kNN search API has been replaced by the `knn` option in the search API.
  res = es_processor.client.knn_search(index="audio_features_3", knn=query_3, _source=["video_id", "start_time", "end_time"])
/tmp/ipykernel_30/3013669256.py:67: GeneralAvailabilityWarning: This API is in technical preview and may be changed or removed in a future release. Elastic will work to fix any issues, but features in technical preview are not subject to the support SLA of official GA features.
  res = es_processor.client.knn_search(index="audio_features_4", knn=query_4, _source=["video_id", "start_time", "end_time"])


Video ID: L01_V028, Start: 497.0, End: 535.0
Video ID: L22_V001, Start: 904.0, End: 936.0
Video ID: L12_V013, Start: 1137.0, End: 1181.0
Video ID: L17_V026, Start: 901.0, End: 953.0
Video ID: L12_V018, Start: 915.0, End: 959.0
Video ID: L07_V030, Start: 1054.0, End: 1108.0
Video ID: L19_V026, Start: 450.0, End: 506.0
Video ID: L10_V002, Start: 958.0, End: 1014.0
Video ID: L11_V005, Start: 539.0, End: 587.0
Video ID: L09_V014, Start: 777.0, End: 825.0
Fourth_part:
Video ID: L01_V005, Start: 434.0, End: 476.0
Video ID: L06_V010, Start: 1138.0, End: 1191.0
Video ID: L15_V021, Start: 501.0, End: 553.0
Video ID: L21_V007, Start: 348.0, End: 412.0
Video ID: L21_V030, Start: 413.0, End: 466.0
Video ID: L14_V026, Start: 1147.0, End: 1197.0
Video ID: L08_V001, Start: 1114.0, End: 1140.0
Video ID: L11_V011, Start: 472.0, End: 519.0
Video ID: L10_V005, Start: 1167.0, End: 1212.0
Video ID: L13_V014, Start: 380.0, End: 420.0


/tmp/ipykernel_30/3013669256.py:67: ElasticsearchWarning: The kNN search API has been replaced by the `knn` option in the search API.
  res = es_processor.client.knn_search(index="audio_features_4", knn=query_4, _source=["video_id", "start_time", "end_time"])
